In [4]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import joblib
import re

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# ============================================================
# FONCTIONS FEATURES
# ============================================================

UNIT_PATTERN = r"(cm|mm|m|kg|g|mg|l|ml|cl|w|kw|v|mah|ah|hz|ghz|mhz|go|gb|to|tb|mp|px|fps|°c|°)"

def get_designation(X):
    return X["designation"].fillna("").astype(str)

def get_description(X):
    return X["description"].fillna("").astype(str)

def first_words_series(X, n=3):
    return (
        X["designation"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.split()
        .str[:n]
        .str.join(" ")
    )

def numbers_units_series(X):
    return (
        X["designation"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.findall(rf"\b\d+[.,]?\d*\s?{UNIT_PATTERN}\b")
        .str.join(" ")
    )

# ============================================================
# 1) Chargement des données
# ============================================================

# Features
X_path = r"C:\Users\Mproo\Documents\Cours_DATASCIENTEST\FEV26-CMLOPS-RAKUTEN\data\raw\X_train_update.csv"
# Target
Y_path = r"C:\Users\Mproo\Documents\Cours_DATASCIENTEST\FEV26-CMLOPS-RAKUTEN\data\processed\Y_train_encode.csv"

X = pd.read_csv(X_path)
Y = pd.read_csv(Y_path)

# Jointure sur "Unnamed: 0"
df = X.merge(Y, on="Unnamed: 0")

print("Shape après merge :", df.shape)
print("Colonnes :", df.columns)
df.head()

# ============================================================
# 2) Séparation X / y
# ============================================================

y = df["prdtypecode_encoded"]  # colonne target encodée
X = df.drop(columns=["prdtypecode_encoded"])

# ============================================================
# 3) Train / Test split
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size :", X_train.shape)
print("Test size :", X_test.shape)

# ============================================================
# TF-IDF
# ============================================================

# ----------- WORD TF-IDF -----------

word_tfidf_designation = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    strip_accents="unicode",
    lowercase=True,
    sublinear_tf=True
)

word_tfidf_description = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1,2),
    strip_accents="unicode",
    lowercase=True,
    sublinear_tf=True
)


# ----------- CHAR TF-IDF -----------

char_tfidf_designation = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3,5),
    min_df=2,
    lowercase=True
)

char_tfidf_description = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3,6),
    min_df=3,
    lowercase=True
)

# ============================================================
# COLUMN TRANSFORMER
# ============================================================

features = ColumnTransformer([

 ("designation_word",
 Pipeline([
     ("select", FunctionTransformer(get_designation, validate=False)),
     ("tfidf", word_tfidf_designation)
 ]),
 ["designation"]),

("designation_char",
 Pipeline([
     ("select", FunctionTransformer(get_designation, validate=False)),
     ("tfidf", char_tfidf_designation)
 ]),
 ["designation"]),

("description_word",
 Pipeline([
     ("select", FunctionTransformer(get_description, validate=False)),
     ("tfidf", word_tfidf_description)
 ]),
 ["description"]),

("description_char",
 Pipeline([
     ("select", FunctionTransformer(get_description, validate=False)),
     ("tfidf", char_tfidf_description)
 ]),
 ["description"]),

    ("first_words",
     Pipeline([
         ("extract", FunctionTransformer(first_words_series, validate=False)),
         ("tfidf", TfidfVectorizer())
     ]),
     ["designation"]),

    ("numbers_units",
     Pipeline([
         ("extract", FunctionTransformer(numbers_units_series, validate=False)),
         ("tfidf", TfidfVectorizer())
     ]),
     ["designation"]),
])

# ============================================================
# PIPELINE FINAL
# ============================================================

pipe = Pipeline([
    ("features", features),
    ("clf", LinearSVC(
        C=1.5,
        class_weight="balanced",
        max_iter=20000
    ))
])

# ============================================================
# TRAINING
# ============================================================

print("==> Entraînement du modèle...")

pipe.fit(X_train, y_train)

print("Training terminé")

# ============================================================
# PREDICTIONS
# ============================================================

print("==> Prédictions...")

y_pred = pipe.predict(X_test)

# ============================================================
# METRICS
# ============================================================

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print("Accuracy :", accuracy)
print("F1 weighted :", f1)

print("\nClassification Report")
print(classification_report(y_test, y_pred))

# ============================================================
# SAVE MODEL
# ============================================================

#model_path = r"C:\Users\Mproo\Documents\Cours_DATASCIENTEST\FEV26-CMLOPS-RAKUTEN\models\rakuten_model.pkl"

#joblib.dump(pipe, model_path)

#print("✅ Modèle sauvegardé :", model_path)

Shape après merge : (84916, 8)
Colonnes : Index(['Unnamed: 0', 'designation', 'description', 'productid', 'imageid',
       'prdtypecode', 'prdtypecode_encoded', 'libelle_type_code'],
      dtype='object')
Train size : (67932, 7)
Test size : (16984, 7)
==> Entraînement du modèle...
Training terminé
==> Prédictions...
Accuracy : 0.8651672162034856
F1 weighted : 0.8648243887283537

Classification Report
              precision    recall  f1-score   support

           0       0.59      0.61      0.60       623
           1       0.78      0.76      0.77       502
           2       0.86      0.87      0.87       336
           3       0.90      0.87      0.88       166
           4       0.81      0.85      0.83       534
           5       0.94      0.97      0.95       791
           6       0.81      0.68      0.74       153
           7       0.80      0.77      0.78       974
           8       0.69      0.65      0.67       414
           9       0.97      0.98      0.97      1009


F1 weighted : 0.8673738961380667